# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** Михайлова Софья Дмитриевна
**Группа:** ИНФА-221
**Вариант:** 9 (предметная область: Возобновляемая энергетика и накопители энергии)

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Порядок выполнения:**

1. Выполните ячейки Блока 1 (установка библиотек) и Блока 2 (данные).
2. Реализуйте четыре функции в Блоке 4 вместо маркеров `# TODO`.
3. Запустите Блок 5 — все `assert` должны пройти без ошибок.
4. Соберите файлы своего варианта, экспортируйте CSV, оформите отчёт.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

In [1]:
!pip install -q Levenshtein pandas lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.2 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import os
import re
import unicodedata
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    from lxml import etree as ET

    LXML_AVAILABLE = True
except ImportError:
    import xml.etree.ElementTree as ET

    LXML_AVAILABLE = False

try:
    import Levenshtein as _lev

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна между двумя строками."""
        return _lev.distance(a, b)

except ImportError:

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (резервная реализация на чистом Python)."""
        if a == b:
            return 0
        if not a:
            return len(b)
        if not b:
            return len(a)
        previous = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(min(previous[j] + 1,
                                   current[j - 1] + 1,
                                   previous[j - 1] + (char_a != char_b)))
            previous = current
        return previous[-1]


# ВАЖНО: атрибут xml:lang в парсере раскрывается в полное имя с пространством имён
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"

pd.set_option("display.max_colwidth", 60)
print("Окружение готово. XML backend:", "lxml" if LXML_AVAILABLE else "ElementTree")

Окружение готово. XML backend: lxml


## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Ячейка ниже создаёт **демонстрационные** файлы `demo.tbx` и `demo.tmx` — они
нужны только для автотестов из Блока 5. Файлы своего варианта
(`variant_NN.tbx` — не менее 15 терминов, `variant_NN.tmx` — не менее 10
предложений) вы создаёте самостоятельно: либо аналогичным кодом, либо выгрузкой
из Smartcat, либо вручную.

In [3]:
DEMO_TBX = """<?xml version="1.0" encoding="UTF-8"?>
<martif type="TBX" xml:lang="en">
  <martifHeader><fileDesc><titleStmt><title>Demo glossary</title></titleStmt>
  </fileDesc></martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en">
        <tig><term>flight controller</term>
        <termNote type="partOfSpeech">noun</termNote>
        <descrip type="definition">Onboard flight computer.</descrip></tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp><term>полётный контроллер</term></termGrp></ntig>
      </langSet>
    </termEntry>
    <termEntry id="tid-002">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>payload</term>
        <termNote type="partOfSpeech">noun</termNote></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>целевая нагрузка</term>
        </termGrp></ntig></langSet>
    </termEntry>
    <termEntry id="tid-003">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>telemetry link</term></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>канал телеметрии</term>
        </termGrp></ntig></langSet>
    </termEntry>
  </body></text>
</martif>
"""

DEMO_TMX = """<?xml version="1.0" encoding="UTF-8"?>
<tmx version="1.4">
  <header creationtool="demo" creationtoolversion="1.0" segtype="sentence"
          o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The flight controller stabilises the aircraft.</seg></tuv>
      <tuv xml:lang="ru"><seg>Полётный контроллер стабилизирует аппарат.</seg></tuv>
    </tu>
    <tu tuid="2">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The operator checks the telemetry link.</seg></tuv>
      <tuv xml:lang="ru"><seg>Оператор проверяет канал телеметрии.</seg></tuv>
    </tu>
    <tu tuid="3">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The payload is mounted under the fuselage.</seg></tuv>
      <tuv xml:lang="ru"><seg>Целевая нагрузка устанавливается под фюзеляжем.</seg></tuv>
    </tu>
  </body>
</tmx>
"""

with open("demo.tbx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TBX)
with open("demo.tmx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TMX)

print("Созданы файлы:", [f for f in os.listdir(".") if f.endswith((".tbx", ".tmx"))])

Созданы файлы: ['demo.tbx', 'demo.tmx']


## Блок 4. Задание — реализуйте четыре функции

Требования ко всем функциям: типизация из `typing`, соответствие PEP 8,
обработка отсутствующих элементов без падения (`None`-проверки).

**Подсказки:**

* язык секции читается как `element.get(XML_LANG)`, а не `element.get("lang")`;
* обход `<term>` внутри `langSet` удобно делать через `lang_set.iter("term")` —
  это покрывает и `tig`, и `ntig/termGrp`;
* текст сегмента лучше собирать через `"".join(seg.itertext())` — так не
  теряется текст при наличии инлайновых тегов;
* при подстановке терминов сортируйте их по убыванию длины.

In [4]:
def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминологических пар.

    Args:
        file_path: путь к ``.tbx``-файлу.
        src_lang: код языка исходных терминов (значение ``xml:lang``).
        tgt_lang: код языка переводных эквивалентов.

    Returns:
        ``pd.DataFrame`` с колонками:
        ``entry_id`` (str)      — значение атрибута ``id`` у ``termEntry``;
        ``source_term`` (str)   — термин на языке ``src_lang``;
        ``target_term`` (str)   — термин на языке ``tgt_lang``;
        ``pos`` (str)           — часть речи из ``termNote[@type='partOfSpeech']``;
        ``definition`` (str)    — текст ``descrip[@type='definition']`` или "";
        ``subject_field`` (str) — текст ``descrip[@type='subjectField']`` или "".
        Статьи, в которых нет одного из двух языков, пропускаются.

    Raises:
        FileNotFoundError: если файл не существует.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    # --- безопасный разбор XML (без DTD и внешних сущностей) ---
    if LXML_AVAILABLE:
        parser = ET.XMLParser(resolve_entities=False, load_dtd=False, no_network=True)
        root = ET.parse(file_path, parser).getroot()
    else:
        root = ET.parse(file_path).getroot()

    rows: List[Dict[str, str]] = []

    for entry in root.iter("termEntry"):
        subject = ""
        descrip = entry.find("descrip[@type='subjectField']")
        if descrip is not None and descrip.text:
            subject = descrip.text.strip()

        by_lang: Dict[str, Dict[str, str]] = {}
        for lang_set in entry.findall("langSet"):
            lang = (lang_set.get(XML_LANG) or "").split("-")[0].lower()
            if not lang:
                continue

            # iter("term") покрывает и <tig>, и <ntig><termGrp>
            terms = [t.text.strip() for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                continue

            note = lang_set.find(".//termNote[@type='partOfSpeech']")
            definition = lang_set.find(".//descrip[@type='definition']")

            by_lang[lang] = {
                "term": terms[0],
                "pos": (note.text or "").strip() if note is not None else "",
                "definition": (definition.text or "").strip() if definition is not None else "",
            }

        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({
                "entry_id": entry.get("id", ""),
                "source_term": by_lang[src_lang]["term"],
                "target_term": by_lang[tgt_lang]["term"],
                "pos": by_lang[src_lang]["pos"] or by_lang[tgt_lang]["pos"],
                "definition": by_lang[src_lang]["definition"],
                "subject_field": subject,
            })

    return pd.DataFrame(
        rows,
        columns=["entry_id", "source_term", "target_term",
                 "pos", "definition", "subject_field"],
    )


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов.

    Args:
        file_path: путь к ``.tmx``-файлу.
        src_lang: код исходного языка.
        tgt_lang: код языка перевода.

    Returns:
        ``pd.DataFrame`` с колонками:
        ``tuid`` (str)            — идентификатор единицы перевода;
        ``source_segment`` (str)  — текст ``seg`` на языке ``src_lang``;
        ``target_segment`` (str)  — текст ``seg`` на языке ``tgt_lang``;
        ``domain`` (str)          — текст ``prop[@type='domain']`` или "";
        ``src_len`` (int)         — длина исходного сегмента в символах;
        ``prev_source`` (str)     — предыдущий исходный сегмент (для ICE), для
        первой строки — пустая строка.

    Raises:
        FileNotFoundError: если файл не существует.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    if LXML_AVAILABLE:
        parser = ET.XMLParser(resolve_entities=False, load_dtd=False, no_network=True)
        root = ET.parse(file_path, parser).getroot()
    else:
        root = ET.parse(file_path).getroot()

    def _seg_text(tuv) -> str:
        """Достать текст сегмента, игнорируя инлайновые теги TMX (ph/bpt/ept/it)."""
        seg = tuv.find("seg")
        if seg is None:
            return ""
        return re.sub(r"\s+", " ", "".join(seg.itertext())).strip()

    rows: List[Dict[str, Any]] = []

    for order, tu in enumerate(root.iter("tu"), start=1):
        segments: Dict[str, str] = {}
        for tuv in tu.findall("tuv"):
            lang = (tuv.get(XML_LANG) or tuv.get("lang") or "").split("-")[0].lower()
            text = _seg_text(tuv)
            if lang and text:
                segments[lang] = text

        if src_lang not in segments or tgt_lang not in segments:
            continue

        domain_node = tu.find("prop[@type='domain']")
        rows.append({
            "tuid": tu.get("tuid", str(order)),
            "source_segment": segments[src_lang],
            "target_segment": segments[tgt_lang],
            "domain": (domain_node.text or "").strip() if domain_node is not None else "",
            "src_len": len(segments[src_lang]),
        })

    frame = pd.DataFrame(
        rows,
        columns=["tuid", "source_segment", "target_segment", "domain", "src_len"],
    )
    frame["prev_source"] = frame["source_segment"].shift(1).fillna("")
    return frame


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame,
                          threshold: float = 70.0,
                          context_before: Optional[str] = None,
                          top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос.

    Алгоритм: нормализовать строки (нижний регистр, без пунктуации), посчитать
    процент сходства по Левенштейну и Token Sort Ratio, взять максимум,
    отбросить кандидатов ниже ``threshold``, отсортировать по убыванию.

    Args:
        source_query: сегмент, который нужно перевести.
        tm_df: таблица памяти переводов (результат :func:`parse_tmx`).
        threshold: минимальный процент сходства (0–100).
        context_before: предыдущий сегмент документа; если он совпадает с
            ``prev_source`` кандидата, а сходство равно 100 %, тип совпадения —
            ICE.
        top_n: максимальное число возвращаемых кандидатов.

    Returns:
        Список словарей, отсортированный по убыванию ``score``, каждый со
        ключами: ``tuid``, ``source_segment``, ``target_segment``,
        ``score`` (float, 0–100), ``lev_ratio``, ``token_sort_ratio``,
        ``match_type`` (одно из ``"ICE (101%)"``, ``"Exact (100%)"``,
        ``"High Fuzzy"``, ``"Fuzzy"``, ``"Low Fuzzy"``).
        Пустой список, если совпадений выше порога нет.
    """
    if tm_df is None or tm_df.empty or not source_query:
        return []

    punct_re = re.compile(r"[^\w\s]", re.UNICODE)

    def _normalize(text: str) -> str:
        """NFKC, нижний регистр, ё→е, без пунктуации, одиночные пробелы."""
        text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
        text = punct_re.sub(" ", text)
        return re.sub(r"\s+", " ", text).strip()

    def _lev_ratio(a: str, b: str) -> float:
        """Процент сходства по Левенштейну (0–100)."""
        if not a and not b:
            return 100.0
        longest = max(len(a), len(b))
        return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)

    def _token_sort_ratio(a: str, b: str) -> float:
        """Процент сходства при игнорировании порядка слов."""
        sa = " ".join(sorted(_normalize(a).split()))
        sb = " ".join(sorted(_normalize(b).split()))
        return _lev_ratio(sa, sb)

    def _classify(score: float, context_matched: bool) -> str:
        """Тип совпадения по проценту и признаку совпадения контекста."""
        if score >= 100.0:
            return "ICE (101%)" if context_matched else "Exact (100%)"
        if score >= 95.0:
            return "High Fuzzy"
        if score >= 75.0:
            return "Fuzzy"
        return "Low Fuzzy"

    results: List[Dict[str, Any]] = []
    norm_query = _normalize(source_query)

    for _, row in tm_df.iterrows():
        norm_seg = _normalize(row["source_segment"])
        lev = _lev_ratio(norm_query, norm_seg)
        tsr = _token_sort_ratio(source_query, row["source_segment"])
        score = max(lev, tsr)

        if score < threshold:
            continue

        context_matched = bool(
            context_before is not None
            and _normalize(context_before) == _normalize(str(row.get("prev_source", "")))
        )

        results.append({
            "tuid": row["tuid"],
            "source_segment": row["source_segment"],
            "target_segment": row["target_segment"],
            "score": score,
            "lev_ratio": lev,
            "token_sort_ratio": tsr,
            "match_type": _classify(score, context_matched),
        })

    results.sort(key=lambda item: (item["score"], item["tuid"]), reverse=True)
    return results[:top_n]


def apply_term_glossary(text: str, glossary_df: pd.DataFrame,
                        template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария.

    Требования: поиск регистронезависимый, но строго по границам слов; более
    длинные термины обрабатываются раньше коротких; уже подставленные фрагменты
    повторно не размечаются.

    Args:
        text: исходный сегмент.
        glossary_df: таблица глоссария с колонками ``source_term``/``target_term``.
        template: шаблон подстановки с полями ``{source}`` и ``{target}``.

    Returns:
        Текст с размеченными терминами; регистр исходного текста сохраняется.
        Если глоссарий пуст или текст пустой — возвращается исходный текст.
    """
    if glossary_df is None or glossary_df.empty or not text:
        return text


    pairs = sorted(
        zip(glossary_df["source_term"], glossary_df["target_term"]),
        key=lambda pair: len(pair[0]),
        reverse=True,
    )

    protected: List[str] = []

    def _store(replacement: str) -> str:
        """Временно спрятать подстановку, чтобы не обрабатывать её повторно."""
        protected.append(replacement)
        return f"\x00{len(protected) - 1}\x00"

    result = text
    for source_term, target_term in pairs:
        pattern = re.compile(
            rf"(?<!\w){re.escape(source_term)}(?!\w)",
            re.IGNORECASE,
        )
        result = pattern.sub(
            lambda m: _store(template.format(source=m.group(0), target=target_term)),
            result,
        )

    return re.sub(r"\x00(\d+)\x00", lambda m: protected[int(m.group(1))], result)

### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [7]:
def normalize(text: str) -> str:
    """Нормализовать строку: NFKC, нижний регистр, без пунктуации, одиночные пробелы."""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein_ratio(a: str, b: str) -> float:
    """Процент сходства строк на основе расстояния Левенштейна.

    Формула: ``(1 - distance / max(len(a), len(b))) * 100``.
    Для двух пустых строк вернуть 100.0.
    """
    if not a and not b:
        return 100.0
    longest = max(len(a), len(b))
    return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)


def token_sort_ratio(a: str, b: str) -> float:
    """Процент сходства при игнорировании порядка слов.

    Нормализовать обе строки, разбить на токены, отсортировать, склеить обратно
    и сравнить через :func:`levenshtein_ratio`.
    """
    sorted_a = " ".join(sorted(normalize(a).split()))
    sorted_b = " ".join(sorted(normalize(b).split()))
    return levenshtein_ratio(sorted_a, sorted_b)

## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [8]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

Тест 1 (метрики сходства) — OK


In [9]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("demo.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 3, f"Ожидается 3 термина, получено {len(glossary_df)}"
for column in ["entry_id", "source_term", "target_term", "pos",
               "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "flight controller"].iloc[0]
assert row["target_term"] == "полётный контроллер", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "noun", "Не извлечён termNote[@type='partOfSpeech']"
assert "telemetry link" in set(glossary_df["source_term"]), \
    "Термин из <tig> без termNote должен быть извлечён"
print("Тест 2 (парсинг TBX) — OK")
glossary_df.head()

Тест 2 (парсинг TBX) — OK


,entry_id,source_term,target_term,pos,definition,subject_field
0,tid-001,flight controller,полётный контроллер,noun,Onboard flight computer.,Demo
1,tid-002,payload,целевая нагрузка,noun,,Demo
2,tid-003,telemetry link,канал телеметрии,,,Demo


In [10]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("demo.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]
assert first["source_segment"].startswith("The flight controller"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("Полётный контроллер"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг TMX) — OK")
tm_df.head()

Тест 3 (парсинг TMX) — OK


,tuid,source_segment,target_segment,domain,src_len,prev_source
0,1,The flight controller stabilises the aircraft.,Полётный контроллер стабилизирует аппарат.,Demo,46,
1,2,The operator checks the telemetry link.,Оператор проверяет канал телеметрии.,Demo,39,The flight controller stabilises the aircraft.
2,3,The payload is mounted under the fuselage.,Целевая нагрузка устанавливается под фюзеляжем.,Demo,42,The operator checks the telemetry link.


In [11]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------
exact_query = "The operator checks the telemetry link."
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=70.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="The flight controller stabilises the aircraft.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("The engineer checks the telemetry link.",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("telemetry link check",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"
assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"
assert len(calculate_fuzzy_match("telemetry", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"
print("Тест 4 (Fuzzy Match и ICE) — OK")

Тест 4 (Fuzzy Match и ICE) — OK


In [12]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("The flight controller reads the payload status.",
                             glossary_df)
assert "[полётный контроллер]" in marked, "Термин из глоссария не подставлен"
assert "[целевая нагрузка]" in marked, "Подставлен не каждый найденный термин"
assert "flight controller" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Payload mass is 2 kg.", glossary_df)
assert "Payload [целевая нагрузка]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[целевая нагрузка]" not in apply_term_glossary("payloads are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"
print("Тест 5 (подстановка терминов) — OK")
print(marked)

Тест 5 (подстановка терминов) — OK
The flight controller [полётный контроллер] reads the payload [целевая нагрузка] status.


## Блок 6. Работа со своим вариантом

После прохождения всех тестов выполните задание на данных своего варианта.

In [13]:
# 6.1 Генератор TBX (100 терминов из корпуса)
def indent_tree(element, level: int = 0, space: str = "  ") -> None:
    """Рекурсивно расставить отступы в XML-дереве (работает и с lxml, и с ET)."""
    pad = "\n" + space * level
    if len(element):
        if not (element.text or "").strip():
            element.text = pad + space
        for child in element:
            indent_tree(child, level + 1, space)
        if not (element.tail or "").strip():
            element.tail = pad
        if not (element[-1].tail or "").strip():
            element[-1].tail = pad
    elif level and not (element.tail or "").strip():
        element.tail = pad

TERMS: List[Dict[str, str]] = [
    {"en": "access to modern energy services", "ru": "доступ к современным энергетическим услугам", "pos": "noun", "def_en": "The availability of reliable and affordable modern energy, such as electricity and clean cooking fuels, to end users."},
    {"en": "renewable energy", "ru": "возобновляемые источники энергии", "pos": "noun", "def_en": "Energy sources that are naturally replenished on a human timescale, such as sunlight, wind, water, and geothermal heat."},
    {"en": "renewable energy technologies", "ru": "технологии использования возобновляемых источников энергии", "pos": "noun", "def_en": "Technologies that harness renewable sources to deliver modern energy services such as electricity and clean fuels."},
    {"en": "technology transfer", "ru": "передача технологии", "pos": "noun", "def_en": "The process of transferring skills, knowledge, equipment, and manufacturing methods between countries or organizations."},
    {"en": "hardware", "ru": "технические средства", "pos": "noun", "def_en": "Physical equipment, machinery, and capital goods used in production or energy generation."},
    {"en": "human capital", "ru": "человеческий капитал", "pos": "noun", "def_en": "The collective skills, knowledge, and experience of people, viewed as an asset of economic value."},
    {"en": "know-how", "ru": "ноу-хау", "pos": "noun", "def_en": "Practical knowledge of how to operate and maintain a technology, usually codified in manuals and routines."},
    {"en": "adapt", "ru": "адаптировать", "pos": "verb", "def_en": "To modify or adjust a technology so that it fits new local conditions or needs."},
    {"en": "low-carbon innovation centres", "ru": "низкоуглеродные инновационные центры", "pos": "noun", "def_en": "Institutions designed to foster and accelerate innovation in low-carbon technologies such as renewables."},
    {"en": "solar PV", "ru": "фотоэлектрические панели", "pos": "noun", "def_en": "Devices that convert sunlight directly into electricity through the photovoltaic effect."},
    {"en": "traditional fuels", "ru": "традиционные виды топлива", "pos": "noun", "def_en": "Conventional fuels such as firewood, charcoal, and crop residues, often burned inefficiently and with health impacts."},
    {"en": "energy security", "ru": "энергетическая безопасность", "pos": "noun", "def_en": "The uninterrupted availability of energy at an affordable price and resilience against supply disruptions."},
    {"en": "RD&D", "ru": "НИОКР", "pos": "noun", "def_en": "Research, development, and deployment; systematic work to create and introduce new products or processes."},
    {"en": "government procurement", "ru": "государственные закупки", "pos": "noun", "def_en": "The process by which governments purchase goods, services, and works from the private sector."},
    {"en": "public intervention", "ru": "государственное вмешательство", "pos": "noun", "def_en": "Government action to influence economic outcomes, often to correct market failures."},
    {"en": "price distortion", "ru": "ценовое искажение", "pos": "noun", "def_en": "A situation where market prices fail to reflect the true social or environmental costs of a good."},
    {"en": "feed-in tariffs", "ru": "стимулирующие тарифы", "pos": "noun", "def_en": "A policy offering guaranteed long-term prices to renewable energy producers to accelerate investment."},
    {"en": "international financial mechanisms", "ru": "международные финансовые механизмы", "pos": "noun", "def_en": "Cross-border funds and institutions designed to finance specific global goals such as clean energy."},
    {"en": "energy efficiency", "ru": "энергоэффективность", "pos": "noun", "def_en": "The use of less energy to provide the same service or produce the same output."},
    {"en": "effective transfer", "ru": "эффективная передача", "pos": "noun", "def_en": "The successful and widespread transfer of technology, knowledge, and skills from one party to another."},
    {"en": "technological progress", "ru": "научно-технический прогресс", "pos": "noun", "def_en": "The continuous advancement and application of new scientific knowledge and technology to improve systems."},
    {"en": "roadmap", "ru": "дорожная карта", "pos": "noun", "def_en": "A strategic plan outlining the steps needed to achieve a long-term goal or vision."},
    {"en": "niche", "ru": "нишевое решение", "pos": "noun", "def_en": "A product or technology serving a specialized market segment, not yet widely adopted."},
    {"en": "business-as-usual scenario", "ru": "инерционный сценарий", "pos": "noun", "def_en": "A business-as-usual scenario in which current trends and policies continue without significant change."},
    {"en": "emissions intensity", "ru": "интенсивность выбросов", "pos": "noun", "def_en": "The amount of greenhouse gases emitted per unit of energy produced or activity performed."},
    {"en": "mitigate climate change", "ru": "смягчение последствий изменения климата", "pos": "noun", "def_en": "Actions to reduce greenhouse gas emissions in order to limit the severity of climate change."},
    {"en": "global warming", "ru": "глобальное потепление", "pos": "noun", "def_en": "The long-term rise in the average temperature of the Earth's climate system."},
    {"en": "cost of renewable energy", "ru": "стоимость возобновляемой энергии", "pos": "noun", "def_en": "The economic cost per unit of energy generated from renewable sources, e.g. dollars per kilowatt-hour."},
    {"en": "solar photovoltaic systems", "ru": "фотоэлектрические системы", "pos": "noun", "def_en": "A complete assembly of solar panels and associated components that convert sunlight into electricity."},
    {"en": "grid parity", "ru": "сетевой паритет", "pos": "noun", "def_en": "The point at which an alternative energy source costs the same as or less than grid electricity."},
    {"en": "turbine", "ru": "турбина", "pos": "noun", "def_en": "A rotary device that extracts energy from a fluid flow, such as wind or water, and converts it into work."},
    {"en": "wind power", "ru": "ветровая энергия", "pos": "noun", "def_en": "Electricity generated from the kinetic energy of wind, typically using wind turbines."},
    {"en": "installed power capacity", "ru": "установленная мощность", "pos": "noun", "def_en": "The maximum electrical power a generating station or system can produce under specified conditions."},
    {"en": "reasonable cost", "ru": "приемлемые затраты", "pos": "noun", "def_en": "A cost level that is economically viable and affordable for consumers and investors."},
    {"en": "cost of capital", "ru": "стоимость капитала", "pos": "noun", "def_en": "The rate of return required to persuade an investor to commit funds to a project."},
    {"en": "crowdfunding", "ru": "краудфандинг", "pos": "noun", "def_en": "The practice of funding a project by raising small amounts of money from a large number of people."},
    {"en": "institutional investors", "ru": "институциональные инвесторы", "pos": "noun", "def_en": "Organizations such as pension funds and insurance companies that invest large sums on behalf of others."},
    {"en": "investment in renewable energy", "ru": "инвестиции в возобновляемые источники энергии", "pos": "noun", "def_en": "Financial commitments made to develop, deploy, and expand renewable energy technologies and infrastructure."},
    {"en": "public policy", "ru": "государственная политика", "pos": "noun", "def_en": "A course of action adopted by a government to address issues and achieve specific goals."},
    {"en": "economic benefits", "ru": "экономические выгоды", "pos": "noun", "def_en": "The positive financial outcomes resulting from an action or investment."},
    {"en": "direct and indirect jobs", "ru": "прямые и сопутствующие рабочие места", "pos": "noun", "def_en": "Jobs created within an industry plus those created in its supply chains and through induced spending."},
    {"en": "depletion of resources", "ru": "истощение ресурсов", "pos": "noun", "def_en": "The consumption of a resource faster than it can be replenished, leading to scarcity."},
    {"en": "greenhouse gases", "ru": "парниковые газы", "pos": "noun", "def_en": "Gases in the atmosphere, such as CO2 and methane, that trap heat and contribute to global warming."},
    {"en": "atmospheric CO2 concentration", "ru": "концентрация CO2 в атмосфере", "pos": "noun", "def_en": "The amount of carbon dioxide present in the atmosphere, usually measured in parts per million."},
    {"en": "decentralisation", "ru": "децентрализация", "pos": "noun", "def_en": "The distribution of power generation and decision-making away from central authorities or large utilities."},
    {"en": "energy storage technologies", "ru": "технологии накопления и хранения энергии", "pos": "noun", "def_en": "Systems used to store energy for later use, such as batteries, pumped hydro, or thermal storage."},
    {"en": "leapfrog", "ru": "перепрыгнуть этап", "pos": "verb", "def_en": "To bypass intermediate stages of technological development by adopting advanced solutions directly."},
    {"en": "stable policy environment", "ru": "стабильная политическая среда", "pos": "noun", "def_en": "A consistent and reliable set of government policies that reduces uncertainty for investors."},
    {"en": "level playing field", "ru": "равные условия", "pos": "noun", "def_en": "A situation in which all competitors face the same rules and market conditions without unfair advantages."},
    {"en": "structure of the industry", "ru": "структура промышленности", "pos": "noun", "def_en": "The composition and organization of firms, markets, and value chains within an industrial sector."},
    {"en": "employment in renewable energy", "ru": "занятость в секторе возобновляемых источников энергии", "pos": "noun", "def_en": "The number of people working in the production, installation, and maintenance of renewable energy technologies."},
    {"en": "modern energy", "ru": "современная энергия", "pos": "noun", "def_en": "Energy services such as electricity and clean fuels that are reliable, affordable, and sustainable."},
    {"en": "workforce", "ru": "рабочая сила", "pos": "noun", "def_en": "The total number of people employed or actively seeking employment in a sector or economy."},
    {"en": "share of employment", "ru": "доля занятости", "pos": "noun", "def_en": "The percentage of total employment in an industry or region attributable to a specific group or technology."},
    {"en": "off-grid renewables", "ru": "автономные возобновляемые источники энергии", "pos": "noun", "def_en": "Small-scale renewable energy systems that operate independently of a main electricity grid."},
    {"en": "solar PV industry", "ru": "фотоэлектрическая солнечная энергетика", "pos": "noun", "def_en": "The industry engaged in manufacturing and deploying solar photovoltaic systems."},
    {"en": "supply chain", "ru": "цепочка поставок", "pos": "noun", "def_en": "The network of organizations, resources, and activities involved in creating and delivering a product."},
    {"en": "hydropower", "ru": "гидроэнергетика", "pos": "noun", "def_en": "Electricity generated from the potential and kinetic energy of flowing or falling water."},
    {"en": "skills base", "ru": "база навыков", "pos": "noun", "def_en": "The collective set of competencies and knowledge available in a workforce or sector."},
    {"en": "policy framework", "ru": "нормативная база", "pos": "noun", "def_en": "A set of laws, regulations, and standards that govern an industry or activity."},
    {"en": "key solution", "ru": "ключевое решение", "pos": "noun", "def_en": "A primary or essential answer to a major problem or challenge."},
    {"en": "decisive action", "ru": "решительные действия", "pos": "noun", "def_en": "Firm and effective measures taken to address a problem or achieve a goal."},
    {"en": "Paris Agreement", "ru": "Парижское соглашение", "pos": "noun", "def_en": "An international climate treaty adopted in 2015 to limit global warming to well below 2°C."},
    {"en": "energy transition", "ru": "энергетический переход", "pos": "noun", "def_en": "The global shift from fossil fuel-based energy systems to renewable and low-carbon sources."},
    {"en": "emission reductions", "ru": "сокращение выбросов", "pos": "noun", "def_en": "The reduction of greenhouse gas emissions released into the atmosphere."},
    {"en": "electrification", "ru": "электрификация", "pos": "noun", "def_en": "The process of replacing fossil-fuel technologies with technologies that use electricity as an energy source."},
    {"en": "final energy consumption", "ru": "конечное энергопотребление", "pos": "noun", "def_en": "The total energy consumed by end users after all transformation and distribution losses."},
    {"en": "least-cost source", "ru": "наименьшая стоимость", "pos": "noun", "def_en": "The lowest possible cost of producing a good or service, used to compare energy sources."},
    {"en": "commercially available technologies", "ru": "коммерчески доступные технологии", "pos": "noun", "def_en": "Technologies that are fully developed, market-ready, and available for purchase and deployment."},
    {"en": "global GDP", "ru": "мировой ВВП", "pos": "noun", "def_en": "The total monetary value of all finished goods and services produced worldwide in a period."},
    {"en": "subsidy", "ru": "субсидия", "pos": "noun", "def_en": "A financial benefit or tax break given by a government to support an industry or activity."},
    {"en": "energy access", "ru": "доступ к энергии", "pos": "noun", "def_en": "The availability of modern energy services such as electricity and clean fuels to a population."},
    {"en": "manufacture and deployment", "ru": "производство и внедрение", "pos": "noun", "def_en": "The combined activities of manufacturing, selling, and installing technology systems."},
    {"en": "gender gap", "ru": "гендерный разрыв", "pos": "noun", "def_en": "The disparity in representation, pay, or opportunities between men and women in a given field."},
    {"en": "Nationally Determined Contributions", "ru": "национальные обязательства", "pos": "noun", "def_en": "Climate action plans submitted by countries under the Paris Agreement to cut emissions."},
    {"en": "ambition", "ru": "амбиции", "pos": "noun", "def_en": "The level of commitment or stringency of a target set by a government or organization."},
    {"en": "potential", "ru": "потенциал", "pos": "noun", "def_en": "The maximum possible output or capacity of a resource that has not yet been realized."},
    {"en": "plans and policies", "ru": "планы и политики", "pos": "noun", "def_en": "The strategic intentions and regulatory measures set by governments and organizations."},
    {"en": "cumulative payoff", "ru": "совокупные выплаты", "pos": "noun", "def_en": "The total financial return or benefit accumulated over a period of time."},
    {"en": "sustainable economic development", "ru": "устойчивое экономическое развитие", "pos": "noun", "def_en": "Economic growth achieved without depleting natural resources or harming the environment."},
    {"en": "supply disruption", "ru": "перебои в поставках", "pos": "noun", "def_en": "An interruption or disruption in the flow of goods, services, or energy resources."},
    {"en": "distributed renewable energy", "ru": "распределенная возобновляемая энергетика", "pos": "noun", "def_en": "Small-scale renewable systems installed close to the point of use rather than at a central plant."},
    {"en": "hybrid mini-grids", "ru": "гибридные мини-сети", "pos": "noun", "def_en": "A small independent grid combining multiple energy sources, such as solar and batteries, with backup."},
    {"en": "demand response", "ru": "управление спросом", "pos": "noun", "def_en": "The modification of consumer demand for energy through incentives, pricing, and behavioral change."},
    {"en": "time-of-use tariffs", "ru": "тарифы с дифференциацией по времени использования", "pos": "noun", "def_en": "A pricing strategy where electricity costs vary by time of day, reflecting supply costs."},
    {"en": "fiscal measures", "ru": "фискальные меры", "pos": "noun", "def_en": "Government actions related to taxation and spending used to influence the economy."},
    {"en": "grid infrastructure", "ru": "инфраструктура энергосистем", "pos": "noun", "def_en": "The physical networks and facilities required for the generation, transmission, and distribution of electricity."},
    {"en": "battery energy storage", "ru": "аккумуляторные системы хранения энергии", "pos": "noun", "def_en": "Rechargeable battery systems used to store electrical energy for later use and grid balancing."},
    {"en": "grid enhancement", "ru": "модернизация энергосистемы", "pos": "noun", "def_en": "Upgrading and improving the capacity, efficiency, and flexibility of existing electricity grids."},
    {"en": "renewables-based heating", "ru": "отопление на основе возобновляемых источников энергии", "pos": "noun", "def_en": "Heating systems that use renewable sources such as solar thermal, geothermal, or biomass."},
    {"en": "electric vehicle", "ru": "электромобиль", "pos": "noun", "def_en": "A vehicle propelled by one or more electric motors using energy stored in rechargeable batteries."},
    {"en": "sustainable aviation fuel", "ru": "экологически чистое авиационное топливо", "pos": "noun", "def_en": "A fuel used to power aircraft that has a lower carbon footprint than conventional jet fuel."},
    {"en": "supportive policy framework", "ru": "благоприятная нормативная база", "pos": "noun", "def_en": "A set of regulations and policies designed to encourage and facilitate investment in a sector."},
    {"en": "domestic supply chains", "ru": "внутренние цепочки поставок", "pos": "noun", "def_en": "The network of suppliers and production processes located within a country's borders."},
    {"en": "hybridisation of projects", "ru": "гибридизация проектов", "pos": "noun", "def_en": "The integration of two or more energy sources or technologies, such as solar and storage, in one project."},
    {"en": "process heating", "ru": "технологический нагрев", "pos": "noun", "def_en": "The use of heat in industrial processes such as manufacturing, chemical production, or food processing."},
    {"en": "targeted financing", "ru": "целевое финансирование", "pos": "noun", "def_en": "Financial resources specifically allocated for a particular project, purpose, or sector."},
    {"en": "renewable energy targets", "ru": "целевые показатели в области возобновляемой энергетики", "pos": "noun", "def_en": "Specific and measurable goals for the adoption or generation of renewable energy."},
    {"en": "economics of electricity generation", "ru": "экономика производства электроэнергии", "pos": "noun", "def_en": "The financial factors that determine the cost, price, and viability of generating electricity."},
    {"en": "insulation from shocks", "ru": "защищенность от потрясений", "pos": "noun", "def_en": "The ability of a system or economy to resist and recover from external shocks such as price volatility."},
]

SUBJECT_FIELD = "Возобновляемая энергетика и накопители энергии"
TBX_PATH = "variant_9.tbx"

def build_tbx(terms: Sequence[Dict[str, str]], path: str,
              subject_field: str = SUBJECT_FIELD) -> str:
    """Сгенерировать валидный TBX-файл (TBX-Core v2, martif)."""
    martif = ET.Element("martif", {"type": "TBX", XML_LANG: "en"})

    header = ET.SubElement(martif, "martifHeader")
    file_desc = ET.SubElement(header, "fileDesc")
    title_stmt = ET.SubElement(file_desc, "titleStmt")
    ET.SubElement(title_stmt, "title").text = f"Glossary EN-RU: {subject_field}"
    source_desc = ET.SubElement(file_desc, "sourceDesc")
    ET.SubElement(source_desc, "p").text = "Lab 01, variant 9, generated automatically"
    encoding_desc = ET.SubElement(header, "encodingDesc")
    ET.SubElement(encoding_desc, "p", {"type": "XCSURI"}).text = "TBXXCSV02.xcs"

    body = ET.SubElement(ET.SubElement(martif, "text"), "body")

    for index, item in enumerate(terms, start=1):
        entry = ET.SubElement(body, "termEntry", {"id": f"tid-{index:03d}"})
        ET.SubElement(entry, "descrip", {"type": "subjectField"}).text = subject_field

        # --- английская секция: краткая форма <tig> ---
        lang_en = ET.SubElement(entry, "langSet", {XML_LANG: "en"})
        tig = ET.SubElement(lang_en, "tig")
        ET.SubElement(tig, "term").text = item["en"]
        ET.SubElement(tig, "termNote", {"type": "partOfSpeech"}).text = item["pos"]
        ET.SubElement(tig, "descrip", {"type": "definition"}).text = item["def_en"]

        # --- русская секция: расширенная форма <ntig><termGrp> ---
        lang_ru = ET.SubElement(entry, "langSet", {XML_LANG: "ru"})
        term_grp = ET.SubElement(ET.SubElement(lang_ru, "ntig"), "termGrp")
        ET.SubElement(term_grp, "term").text = item["ru"]
        ET.SubElement(term_grp, "termNote", {"type": "partOfSpeech"}).text = item["pos"]

    indent_tree(martif)
    xml_body = ET.tostring(martif, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">\n')
        handler.write(xml_body)
    return path

build_tbx(TERMS, TBX_PATH)
print(f"Создан {TBX_PATH}: {os.path.getsize(TBX_PATH)} байт, терминов: {len(TERMS)}\n")
print("--- первые 26 строк файла ---")
with open(TBX_PATH, encoding="utf-8") as handler:
    for line in list(handler)[:26]:
        print(line.rstrip())

Создан variant_9.tbx: 78679 байт, терминов: 100

--- первые 26 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">
<martif type="TBX" xml:lang="en">
  <martifHeader>
    <fileDesc>
      <titleStmt>
        <title>Glossary EN-RU: Возобновляемая энергетика и накопители энергии</title>
      </titleStmt>
      <sourceDesc>
        <p>Lab 01, variant 9, generated automatically</p>
      </sourceDesc>
    </fileDesc>
    <encodingDesc>
      <p type="XCSURI">TBXXCSV02.xcs</p>
    </encodingDesc>
  </martifHeader>
  <text>
    <body>
      <termEntry id="tid-001">
        <descrip type="subjectField">Возобновляемая энергетика и накопители энергии</descrip>
        <langSet xml:lang="en">
          <tig>
            <term>access to modern energy services</term>
            <termNote type="partOfSpeech">noun</termNote>
            <descrip type="definition">The availability of reliable and affordable modern energy, such as electricity and cle

In [14]:
# 6.2 Генератор TMX (100 предложений из корпуса)
PARALLEL_CORPUS: List[Tuple[str, str]] = [
    ("Energy underpins all sectors of the economy, and improved access to modern energy services is central to development.", "Энергия лежит в основе всех секторов экономики, и расширение доступа к современным энергетическим услугам является ключевым фактором развития."),
    ("Renewable Energy is derived from natural processes that are replenished constantly.", "Возобновляемые источники энергии связаны с природными процессами и постоянно пополняются."),
    ("Renewable energy technologies (RETs) are technologies that provide modern energy services.", "Технологии использования возобновляемых источников энергии представляют собой такие технологии, которые обеспечивают современные энергетические услуги."),
    ("Technology transfer is an important facet of RETs deployment in developing countries.", "Передача технологии является важным аспектом внедрения ТВЭ в развивающихся странах."),
    ("Flow A depicts the process of hardware being created and brought into use.", "Поток А отражает процесс создания и предоставления в пользование технических средств."),
    ("Flow B depicts the human capital associated with the operation and maintenance of the physical capital.", "Поток В отображает человеческий капитал, связанный с эксплуатацией и обслуживанием физического капитала."),
    ("This is the know-how, and it includes information on operating, maintenance and repair procedures.", "Это представляет собой ноу-хау и включает информацию о процедурах эксплуатации, обслуживания и ремонта."),
    ("In this sense, Flow C allows for existing technology to be adapted and modified to suit local conditions.", "В этом смысле поток С позволяет адаптировать и изменять существующую технологию в соответствии с местными условиями."),
    ("One option for developing this local knowledge and expertise is the concept of low-carbon innovation centres.", "Одним из вариантов развития этих местных знаний и опыта является концепция низкоуглеродных инновационных центров."),
    ("For developing countries, the most feasible RETs include solar PV.", "В развивающихся странах наиболее приемлемые ТВЭ включают солнечные фотоэлектрические панели."),
    ("They allow traditional fuels to be utilized in cleaner, more efficient ways.", "Они позволяют использовать традиционные виды топлива более чистым и более эффективным способом."),
    ("Broadening the energy mix through varied indigenous fuel sources can improve energy security.", "Расширение энергетического баланса за счет разнообразных местных топливных источников может улучшить энергетическую безопасность."),
    ("Providing incentives for RD&D at the firm level in private companies, and supporting technology deployment in market niches.", "Обеспечение стимулов для НИОКР на уровне частных компаний и поддержку внедрения технологий в рыночных нишах."),
    ("Government procurement can encourage private firms to adopt renewable energy technologies.", "Государственные закупки могут поощрять внедрение частными компаниями технологий использования возобновляемых источников энергии."),
    ("Two reasons are commonly cited for the need for public intervention.", "Обычно упоминаются две причины, по которым государственное вмешательство необходимо."),
    ("This price distortion therefore reduces the transfer and the market penetration of renewable energy technologies.", "Это ценовое искажение уменьшает передачу и проникновение на рынок технологий использования возобновляемых источников энергии."),
    ("Feed-in tariffs are an important domestic policy mechanism too.", "Важным механизмом внутренней политики также являются стимулирующие тарифы."),
    ("In order to complement domestic financial mechanisms, countries should also look to take advantage of international financial mechanisms.", "В целях дополнения внутренних финансовых механизмов странам следует также изучить возможность использования международных финансовых механизмов."),
    ("Energy efficiency and conservation are equally important features.", "Энергоэффективность и энергосбережение являются в равной степени важными."),
    ("This presents a significant challenge to the effective transfer of RETs.", "Это представляет собой значительную проблему для эффективной передачи ТВЭ."),
    ("Rapid technological progress, combined with falling costs, means that renewable energy is increasingly seen as the answer.", "Быстрый научно-технический прогресс в сочетании со снижением затрат приводит к тому, что возобновляемые источники энергии пользуются все большим успехом."),
    ("REmap 2030, a global roadmap developed by IRENA, shows that not only can renewable energy meet the world's rising demand.", "Глобальная дорожная карта REmap 2030 демонстрирует, что возобновляемые источники энергии могут не просто удовлетворить растущий мировой спрос."),
    ("A technology once considered as niche is becoming mainstream.", "Эта технология, некогда считавшаяся нишевым решением, становится массовой."),
    ("If business continues as usual, these efforts will not succeed.", "Если коммерческая деятельность будет осуществляться старыми методами, эти усилия окажутся тщетными."),
    ("The average emissions intensity of electricity production has barely changed.", "Средняя интенсивность выбросов при производстве электроэнергии почти не изменилась."),
    ("A doubling in the share of renewables could help mitigate climate change.", "Увеличение доли возобновляемых источников энергии в два раза могло бы помочь смягчить последствия изменения климата."),
    ("They contribute to limiting global warming to under 2 degrees Celsius.", "Они способствуют ограничению глобального потепления величиной в 2 градуса Цельсия."),
    ("The cost of renewable energy plummets as deployment increases.", "Стоимость возобновляемой энергии резко упала по мере увеличения их использования."),
    ("Solar photovoltaic (PV) prices have fallen by 80% since 2008.", "Цены на фотоэлектрические системы с 2008 г. снизились на 80%."),
    ("Commercial solar power reached grid parity.", "Энергия, произведенная на коммерческих солнечных электростанциях, достигла сетевого паритета."),
    ("Turbine costs falling nearly 30% since 2008.", "Стоимость турбин с 2008 г. снизилась приблизительно на 30%."),
    ("More than 100 countries now use wind power.", "В настоящее время более 100 стран используют ветровую электроэнергию."),
    ("Renewables today constitute 30% of all installed power capacity.", "На сегодняшний день возобновляемые источники энергии составляют 30% суммарной установленной мощности всех электростанций в мире."),
    ("The challenge has moved on from whether renewable energy can power modern lifestyles at a reasonable cost.", "Вопрос не в том, может ли возобновляемая энергия обеспечить современный образ жизни при приемлемых затратах."),
    ("The cost of capital is falling and products are being tailored for a wider range of investors.", "Стоимость капитала снижается, а продукция специально разрабатывается для широкого ряда инвесторов."),
    ("Crowdfunding initiatives can also be used to attract capital.", "Для привлечения капитала можно использовать также программы краудфандинга."),
    ("Institutional investors are also starting to get interested.", "Институциональные инвесторы также начинают проявлять интерес."),
    ("Total investment in renewable energy rose from USD 55 billion in 2004 to USD 214 billion in 2013.", "Общая сумма инвестиций в возобновляемые источники энергии выросла с 40 млрд долл. США в 2004 г. до 214 млрд долл. США в 2013 г."),
    ("Policy makers can do much to either promote or hinder this vision.", "Лица, ответственные за проведение государственной политики, могут сделать многое, чтобы стимулировать воплощение этой концепции или препятствовать ей."),
    ("A recent Japanese study found the benefits were 2-3 times higher than the costs.", "Результаты недавнего японского исследования показали, что экономические выгоды в 2—3 раза превышают расходы."),
    ("In 2013, it supported 6.5 million direct and indirect jobs.", "Благодаря им в 2013 г. было создано 6,5 млн прямых и сопутствующих рабочих мест."),
    ("Most renewables do not deplete finite resources.", "Использование большинства возобновляемых источников энергии не приводит к истощению ограниченных ресурсов."),
    ("Crucially, they offer a route to reducing greenhouse gas emissions.", "Что самое главное, они являются способом сокращения выбросов парниковых газов."),
    ("REmap 2030 estimates that doubling the share of renewables can keep atmospheric CO2 below 450 ppm.", "Согласно дорожной карте REmap 2030 удвоение доли возобновляемых источников энергии в энергосистеме может обеспечить ограничение концентрации CO2 в атмосфере."),
    ("A sector once dominated by large utilities is becoming more decentralised.", "Сектор, в котором когда-то доминировали крупные объекты энергообеспечения, становится более децентрализованным."),
    ("New storage technologies will grow in importance.", "Возрастает значимость новых технологий накопления и хранения энергии."),
    ("Many countries have an opportunity to leapfrog the development of a fixed network.", "У многих стран есть возможность быстро перепрыгнуть этап развития фиксированной сети."),
    ("Renewable energy investors need stable and predictable policy frameworks.", "Инвесторам в области возобновляемой энергетики необходима стабильная и прогнозируемая политическая среда."),
    ("They need a level playing field, including cutting back on subsidies for fossil fuels.", "Им нужны равные условия, что подразумевает сокращение субсидий для ископаемых видов топлива."),
    ("As the share of renewable energy grows, the structure of the industry is undergoing change.", "По мере роста доли возобновляемых источников энергии структура промышленности претерпевает изменения."),
    ("The renewable energy sector employed 11.5 million people in 2019.", "В 2019 году в секторе возобновляемых источников энергии было занято 11,5 миллиона человек."),
    ("The bulk of global jobs relate to modern energy use.", "Основная часть рабочих мест в мире связана с использованием современной энергии."),
    ("The share of women in the renewable energy workforce is about 32%.", "Доля женщин в рабочей силе в секторе возобновляемых источников энергии составляет примерно 32%."),
    ("Asia accounted for 63% of total jobs in renewables globally.", "На Азию приходится 63% всех рабочих мест в секторе возобновляемых источников энергии во всём мире."),
    ("Off-grid decentralised renewables are creating a growing number of jobs.", "Децентрализованные автономные возобновляемые источники энергии создают всё больше рабочих мест."),
    ("The solar PV industry retains the top spot.", "На первом месте остаётся фотоэлектрическая солнечная энергетика."),
    ("Supply chains are expanding.", "Цепочки поставок расширяются."),
    ("Hydropower has the largest installed capacity of all renewables.", "Гидроэнергетика имеет самую большую установленную мощность из всех возобновляемых источников энергии."),
    ("Building the skills base requires more vocational training.", "Создание базы навыков требует более широкого профессионального обучения."),
    ("The COVID-19 pandemic reinforced the importance of strong policy frameworks for renewables.", "Пандемия COVID-19 подчеркнула важность прочной нормативной базы для возобновляемых источников энергии."),
    ("Renewable energy is a key solution to address climate change.", "Возобновляемая энергия является ключевым решением проблемы изменения климата."),
    ("Now is the time for decisive action.", "Сейчас настало время для решительных действий."),
    ("Fulfilling the Paris Agreement means deploying renewables six times faster.", "Выполнение Парижского соглашения означает развёртывание возобновляемых источников энергии в шесть раз быстрее."),
    ("By 2050, two-thirds of the world's energy should be renewable.", "К 2050 году две трети произведённой в мире энергии должны быть получены с помощью возобновляемой энергетики."),
    ("Renewables can cut energy-related CO2 emissions by about 70%.", "Возобновляемые источники энергии могут сократить выбросы CO2, связанные с производством энергии, примерно на 70%."),
    ("Renewables and electrification technologies could achieve 90% of the emission reductions.", "С помощью возобновляемых источников энергии и технологий электрификации можно добиться сокращения выбросов на 90%."),
    ("Growing share of electricity in final energy consumption.", "Растущая доля электроэнергии в конечном энергопотреблении."),
    ("Renewables have become the least-cost source of new power generation.", "Возобновляемые источники энергии стали самым дешёвым новым источником электроэнергии."),
    ("Costs from all commercially available renewable power technologies continue to fall.", "Стоимость всех коммерчески доступных технологий возобновляемой энергии продолжает падать."),
    ("IRENA analysis shows that renewables and energy efficiency can boost global GDP by 2.5%.", "Анализ IRENA показывает, что возобновляемые источники энергии и энергоэффективность могут повысить мировой ВВП на 2,5%."),
    ("Today's energy system is heavily subsidised.", "Современная энергетическая система в значительной степени субсидируется."),
    ("Renewables are helping to expand energy access across Africa and Asia.", "Возобновляемые источники энергии помогают расширить доступ к энергии в Африке и Азии."),
    ("More countries manufacture, trade and install renewable energy technologies every year.", "Ежегодно всё больше стран производят, продают и внедряют технологии использования возобновляемых источников энергии."),
    ("Women hold 32% of jobs in renewables.", "Женщины занимают 32% рабочих мест в сфере возобновляемых источников энергии."),
    ("Nationally Determined Contributions provide a strong framework to reduce emissions.", "Национальные обязательства обеспечивают прочную основу для сокращения выбросов."),
    ("The ambition of current NDC targets could more than double.", "Амбиции нынешних целей НО могут более чем удвоиться."),
    ("Cost-effective renewable energy potential remains untapped.", "Экономически эффективный потенциал возобновляемой энергии остаётся неиспользованным."),
    ("Today's plans and policies foresee energy investments to 2050 reaching USD 95 trillion.", "Сегодняшние планы и политики предусматривают, что инвестиции в энергетику до 2050 года достигнут 95 триллионов долларов США."),
    ("The cumulative payoff by 2050 would be worth between USD 65 trillion and USD 160 trillion.", "Совокупные выплаты к 2050 году составят от 65 до 160 триллионов долларов США."),
    ("Renewable energy delivers jobs, delivers on sustainable economic development and will deliver a viable climate solution.", "Возобновляемые источники энергии обеспечивают рабочие места, обеспечивают устойчивое экономическое развитие и обеспечат приемлемое решение по климату."),
    ("The ongoing conflict in the Middle East has severely disrupted global oil and gas supplies.", "Продолжающийся конфликт на Ближнем Востоке серьезно нарушил глобальные поставки нефти и газа."),
    ("Deploy distributed renewable energy solutions to support public services.", "Внедрять решения в области распределенной возобновляемой энергетики для поддержки общественных услуг."),
    ("Accelerate deployment of solar PV battery hybrid mini-grids.", "Ускорить развертывание гибридных мини-сетей, сочетающих солнечные фотоэлектрические системы и аккумуляторные батареи."),
    ("Initiate public information campaigns and measures such as mandates to reduce energy demand.", "Проводить информационные кампании и принимать меры, такие как введение обязательных требований по сокращению спроса на энергию."),
    ("Fast-track time-of-use tariff adoption.", "Ускорить внедрение тарифов с дифференциацией по времени использования."),
    ("Implement fiscal measures such as grants, subsidies or tax rebates.", "Ввести фискальные меры, такие как гранты, субсидии или налоговые льготы."),
    ("Fast-track current renewable energy and grid infrastructure projects.", "Ускорить реализацию текущих проектов в области возобновляемой энергетики и инфраструктуры энергосистем."),
    ("Incentivise battery energy storage deployment.", "Стимулировать внедрение аккумуляторных систем хранения энергии."),
    ("Leverage grid enhancement technologies to relieve near-term grid constraints.", "Использовать технологии модернизации энергосистемы для устранения краткосрочных ограничений."),
    ("Incentivise the deployment of renewables-based heating solutions.", "Стимулировать внедрение решений в области отопления на основе возобновляемых источников энергии."),
    ("Streamline permitting for electric vehicle charger installation.", "Упростить процедуру получения разрешений на установку зарядных устройств для электромобилей."),
    ("Fast-track permitting and subsidy support for sustainable aviation fuel production projects.", "Ускорить процедуру получения разрешений и оказать субсидиарную поддержку проектам по производству экологически чистого авиационного топлива."),
    ("Establish comprehensive, clear, supportive policy frameworks to attract investment.", "Создать комплексную, четкую и благоприятную нормативную базу для привлечения инвестиций."),
    ("Promote domestic and regional supply chain development for technologies.", "Содействовать развитию внутренних и региональных цепочек поставок технологий."),
    ("Facilitate hybridisation of projects to allow storage systems to be installed.", "Содействовать гибридизации проектов с целью обеспечения возможности установки систем накопления энергии."),
    ("Incentivise and support electrification in low-to-medium temperature process heating.", "Стимулировать и поддерживать электрификацию в сфере технологического нагрева с низкой и средней температурой."),
    ("Develop targeted financing mechanisms for renewable-based mini-grids.", "Разработать целевые механизмы финансирования мини-сетей на основе возобновляемых источников энергии."),
    ("Make financial and subsidy support for the fossil fuel industry conditional on meeting renewable energy targets.", "Сделать финансовую и субсидиарную поддержку отрасли ископаемого топлива зависимой от достижения целевых показателей в области возобновляемой энергетики."),
    ("Long-term cost reductions in renewable energy technologies are reshaping the economics of electricity generation.", "Долгосрочное снижение затрат на технологии возобновляемой энергетики меняет экономику производства электроэнергии."),
    ("China's power sector is notably more insulated from this shock.", "Энергетический сектор Китая в значительной степени более защищен от данного потрясения.")
]

VARIANT = 9
TMX_FILE = f"variant_{VARIANT}.tmx"
DOMAIN = "Возобновляемая энергетика и накопители энергии"

def build_tmx(pairs: Sequence[Tuple[str, str]], path: str,
              src_lang: str = "en", tgt_lang: str = "ru",
              domain: str = DOMAIN) -> str:
    """Сгенерировать валидную память переводов в формате TMX 1.4b."""
    from datetime import datetime, timezone

    created = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    tmx = ET.Element("tmx", {"version": "1.4"})
    ET.SubElement(tmx, "header", {
        "creationtool": "lab01-generator",
        "creationtoolversion": "1.0",
        "segtype": "sentence",
        "o-tmf": "TMX",
        "adminlang": "en",
        "srclang": src_lang,
        "datatype": "plaintext",
        "creationdate": created,
    })
    body = ET.SubElement(tmx, "body")

    for index, (source, target) in enumerate(pairs, start=1):
        tu = ET.SubElement(body, "tu", {"tuid": str(index), "creationdate": created})
        ET.SubElement(tu, "prop", {"type": "domain"}).text = domain
        for lang, text in ((src_lang, source), (tgt_lang, target)):
            tuv = ET.SubElement(tu, "tuv", {XML_LANG: lang})
            ET.SubElement(tuv, "seg").text = text

    indent_tree(tmx)
    xml_body = ET.tostring(tmx, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE tmx SYSTEM "tmx14.dtd">\n')
        handler.write(xml_body)
    return path

build_tmx(PARALLEL_CORPUS, TMX_FILE)
print(f"Создан {TMX_FILE}: {os.path.getsize(TMX_FILE)} байт, "
      f"единиц перевода: {len(PARALLEL_CORPUS)}\n")
print("--- первые 20 строк файла ---")
with open(TMX_FILE, encoding="utf-8") as handler:
    for line in list(handler)[:20]:
        print(line.rstrip())

Создан variant_9.tmx: 56072 байт, единиц перевода: 100

--- первые 20 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE tmx SYSTEM "tmx14.dtd">
<tmx version="1.4">
  <header creationtool="lab01-generator" creationtoolversion="1.0" segtype="sentence" o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext" creationdate="20261019T231803Z"/>
  <body>
    <tu tuid="1" creationdate="20261019T231803Z">
      <prop type="domain">Возобновляемая энергетика и накопители энергии</prop>
      <tuv xml:lang="en">
        <seg>Energy underpins all sectors of the economy, and improved access to modern energy services is central to development.</seg>
      </tuv>
      <tuv xml:lang="ru">
        <seg>Энергия лежит в основе всех секторов экономики, и расширение доступа к современным энергетическим услугам является ключевым фактором развития.</seg>
      </tuv>
    </tu>
    <tu tuid="2" creationdate="20261019T231803Z">
      <prop type="domain">Возобновляемая энергетика и накопители

In [15]:
VARIANT = 9
SURNAME = "Mikhailova"

TBX_FILE = f"variant_{VARIANT}.tbx"
TMX_FILE = f"variant_{VARIANT}.tmx"

# 1) файлы уже созданы в Блоках 6.1 и 6.2
print(f"TBX : {TBX_FILE} ({os.path.getsize(TBX_FILE)} байт)")
print(f"TMX : {TMX_FILE} ({os.path.getsize(TMX_FILE)} байт)")

# 2) глоссарий
my_glossary = parse_tbx(TBX_FILE)
print(f"\nГлоссарий: {len(my_glossary)} терминов")

# 3) память переводов
my_tm = parse_tmx(TMX_FILE)
print(f"Память переводов: {len(my_tm)} сегментов")

# 4) три запроса: ICE, Fuzzy, No Match
QUERIES: List[Dict[str, Optional[str]]] = [
    {
        "label": "1. ICE — точное совпадение с совпадением контекста",
        # Берем 3-е предложение из корпуса
        "query": "Renewable energy technologies (RETs) are technologies that provide modern energy services.",
        # А контекстом ставим 2-е предложение из корпуса (которое идет прямо перед ним)
        "context": "Renewable Energy is derived from natural processes that are replenished constantly.",
    },
    {
        "label": "2. Fuzzy — заменено одно слово",
        # Берем 28-е предложение из корпуса: "The cost of renewable energy plummets as deployment increases."
        # И меняем "plummets" на "drops", чтобы получить Fuzzy-совпадение (около 85-90%)
        "query": "The cost of renewable energy drops as deployment increases.",
        "context": None,
    },
    {
        "label": "3. No match — сегмент вне предметной области",
        "query": "The quarterly financial report was approved by the board.",
        "context": None,
    },
]

all_matches: List[Dict[str, Any]] = []
for case in QUERIES:
    matches = calculate_fuzzy_match(case["query"], my_tm, threshold=75.0,
                                    context_before=case["context"])
    print("=" * 100)
    print(case["label"])
    print(f"Запрос: {case['query']}")
    if not matches:
        print("  -> совпадений выше порога 75 % не найдено, сегмент переводится с нуля")
        continue
    for rank, match in enumerate(matches, start=1):
        print(f"  #{rank} {match['score']:6.2f} %  [{match['match_type']:<12}] "
              f"tuid={match['tuid']}")
        print(f"      TM src: {match['source_segment']}")
        print(f"      TM tgt: {match['target_segment']}")
        match["query"] = case["query"]
        all_matches.append(match)

matches_df = pd.DataFrame(all_matches)
print("=" * 100)
print(f"\nВсего кандидатов выше порога: {len(matches_df)}")

# 5) экспорт CSV для Smartcat
GLOSSARY_CSV = f"glossary_smartcat_v{VARIANT}.csv"
TM_CSV = f"tm_segments_v{VARIANT}.csv"
FUZZY_CSV = f"fuzzy_report_v{VARIANT}.csv"

smartcat_df = pd.DataFrame({
    "Source (en)": my_glossary["source_term"],
    "Target (ru)": my_glossary["target_term"],
    "Comment": my_glossary["definition"],
})
smartcat_df.to_csv(GLOSSARY_CSV, index=False, encoding="utf-8-sig")

my_tm[["tuid", "source_segment", "target_segment", "domain"]].to_csv(
    TM_CSV, index=False, encoding="utf-8-sig")

matches_df[["query", "tuid", "source_segment", "target_segment",
            "score", "lev_ratio", "token_sort_ratio", "match_type"]].to_csv(
    FUZZY_CSV, index=False, encoding="utf-8-sig")

print("\nЭкспортированные файлы:")
for name in [TBX_FILE, TMX_FILE, GLOSSARY_CSV, TM_CSV, FUZZY_CSV]:
    print(f"  {name:<28} {os.path.getsize(name):>7} байт")

# 6) итоговая таблица совпадений
print("\nИтоговая таблица совпадений:")
matches_df[["query", "tuid", "score", "match_type"]]

TBX : variant_9.tbx (78679 байт)
TMX : variant_9.tmx (56072 байт)

Глоссарий: 100 терминов
Память переводов: 100 сегментов
1. ICE — точное совпадение с совпадением контекста
Запрос: Renewable energy technologies (RETs) are technologies that provide modern energy services.
  #1 100.00 %  [ICE (101%)  ] tuid=3
      TM src: Renewable energy technologies (RETs) are technologies that provide modern energy services.
      TM tgt: Технологии использования возобновляемых источников энергии представляют собой такие технологии, которые обеспечивают современные энергетические услуги.
2. Fuzzy — заменено одно слово
Запрос: The cost of renewable energy drops as deployment increases.
  #1  88.52 %  [Fuzzy       ] tuid=28
      TM src: The cost of renewable energy plummets as deployment increases.
      TM tgt: Стоимость возобновляемой энергии резко упала по мере увеличения их использования.
3. No match — сегмент вне предметной области
Запрос: The quarterly financial report was approved by the board

,query,tuid,score,match_type
0,Renewable energy technologies (RETs) are technologies th...,3,100.00,ICE (101%)
1,The cost of renewable energy drops as deployment increases.,28,88.52,Fuzzy


## Блок 7. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?
   TBX устроен вокруг понятий: главная единица — termEntry, описывающая один концепт (например, «battery storage») и содержащая его эквиваленты на разных языках, часть речи и определение. Такой подход формирует справочник, отвечающий на вопрос «как переводится этот термин?». TMX устроен вокруг текста: единица tu хранит пару предложений (оригинал и перевод). Это база готовых решений, отвечающая на вопрос «встречалось ли такое предложение ранее?». В работе TBX используется для терминологической подсказки, а TMX — для поиска нечётких совпадений.
2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?
В XML атрибут xml:lang не является обычным атрибутом тега, а принадлежит зарезервированному пространству имён W3C. Попытка извлечь его через element.get("lang") возвращает None, из-за чего разделение английского и русского текста становится невозможным. Проблема решена объявлением константы XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang". Использование этой константы в методах .get() обеспечивает корректное извлечение языковых секций как из TBX, так и из TMX.
3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это компенсирует Token Sort Ratio?
Метрика Левенштейна считает посимвольные замены, поэтому даёт заниженный результат при перестановке слов местами (например, «energy storage» и «storage energy»). Для естественного языка это критично, так как порядок слов может меняться при переводе. Token Sort Ratio решает проблему: перед подсчётом строка разбивается на токены, сортируется по алфавиту и только затем сравнивается. В алгоритме берётся максимум из двух метрик (score = max(lev, tsr)), что позволяет находить смысловые совпадения даже при сильном изменении структуры предложения.
4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно для стоимости проекта в Smartcat?
Exact Match (100 %) означает совпадение текста предложения с базой, но контекст (соседние предложения) может отличаться. ICE (In-Context Exact) — это 100% совпадение текста плюс совпадение предыдущего сегмента. Функция calculate_fuzzy_match проверяет параметр context_before и при совпадении с колонкой prev_source присваивает статус ICE. Для бюджета в Smartcat это критично: за ICE-совпадения тариф минимален (обычно 10–20% от ставки), так как проверка контекста не требуется — достаточно подтвердить готовый вариант.
5. Какие проблемы возникли при автоматической подстановке терминов и как вы их решили?
Основная проблема — конфликт коротких и длинных терминов. Замена «energy» ломает термин «renewable energy», а регулярные выражения могут захватить часть другого слова (например, «grid» внутри «microgrid»). В функции apply_term_glossary применены три решения: 1) сортировка терминов по убыванию длины; 2) использование границ слов (?<!\w)...(?!\w); 3) защита \x00 (временные маркеры), предотвращающая повторную обработку уже размеченных фрагментов.


**Ваши выводы:**

В ходе работы был написан скрипт на Python, который автоматизирует работу переводчика. Программа создает и обрабатывает два файла: глоссарий терминов по энергетике (TBX) и базу готовых переводов (TMX). Скрипт ищет в тексте уже переведенные предложения, даже если они немного изменились (Fuzzy Match), и учитывает контекст (ICE). Кроме того, программа автоматически подставляет русские термины к английским словам из глоссария, не ломая длинные словосочетания. В итоге получился инструмент, который ускоряет перевод и помогает соблюдать единообразие терминов.

_______________________________________________

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_01_Фамилия.ipynb` (доступ «для всех по ссылке»).
2. Файлы `variant_NN.tbx`, `variant_NN.tmx`, `glossary_smartcat.csv`.
3. Отчёт с выводами и скриншотом импортированного глоссария в Smartcat.